In [34]:
%pip install numpy
%pip install matplotlib
%pip install pandas
%pip install scipy
%pip install opencv-python
%pip install imageio
!pip uninstall -y opencv-python opencv-contrib-python
!pip install opencv-python-headless
!pip install imageio-ffmpeg
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached opencv_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (73.8 MB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 23.2 MB/s  0:00:01m0:00:0100:01


In [24]:
file_path_D1 = r"Pruebas_DELFT_20260929.txt"
file_path_D2 = r"Pruebas_DELFT_20261002.txt"

columns = [
    "point_id",
    "date",
    "time",
    "easting",
    "northing",
    "ellipsoidal_height",
    "geoid_separation",
    "cq_3d",
]

def load_gps(file_path):
    gps = pd.read_csv(
        file_path,
        sep=r"\s+",
        skiprows=1,          # skip the descriptive header
        names=columns,
        dtype={
            "point_id": "string",
            "date": "string",
            "time": "string",
            "easting": "float64",
            "northing": "float64",
            "ellipsoidal_height": "float64",
            "geoid_separation": "float64",
            "cq_3d": "float64",
        },
    )

    gps["datetime"] = pd.to_datetime(
        gps["date"] + " " + gps["time"],
        format="%m/%d/%Y %H:%M:%S",
    )
    gps["datetime"] = gps["datetime"] - pd.Timedelta(hours=1)
    gps = gps.drop(columns=["date", "time"])

    # Determining actual elevations
    gps["elevation"] = gps["ellipsoidal_height"] - gps["geoid_separation"]
    return gps

gps_D1 = load_gps(file_path_D1)
gps_D2 = load_gps(file_path_D2)


display(gps_D1.head())
display(gps_D2.head())

,point_id,easting,northing,ellipsoidal_height,geoid_separation,cq_3d,datetime,elevation
0,GPS0001,806688.1680,2.343461e+06,-15.2602,-13.9310,0.0159,2026-09-29 11:18:19,-1.3292
1,GPS0002,806687.4266,2.343459e+06,-15.2558,-13.9309,0.0159,2026-09-29 11:18:40,-1.3249
2,GPS0003,806697.2479,2.343461e+06,-15.3033,-13.9310,0.0177,2026-09-29 11:19:18,-1.3723
3,GPS0004,806687.4149,2.343452e+06,-15.3480,-13.9308,0.0156,2026-09-29 11:19:56,-1.4172
4,GPS0005,806680.4073,2.343383e+06,-14.3263,-13.9294,0.0210,2026-09-29 11:22:42,-0.3969


,point_id,easting,northing,ellipsoidal_height,geoid_separation,cq_3d,datetime,elevation
0,RTCM-Ref0017,806520.1897,2.343199e+06,-2.3498,-13.9267,NaN,2026-10-02 09:52:57,11.5769
1,GPS_Auto_000001,806503.2658,2.343120e+06,-14.1150,-13.9252,0.0171,2026-10-02 09:54:30,-0.1898
2,GPS_Auto_000002,806503.4395,2.343120e+06,-14.1546,-13.9252,0.0157,2026-10-02 09:54:31,-0.2294
3,GPS_Auto_000003,806503.7183,2.343120e+06,-14.0603,-13.9252,0.0158,2026-10-02 09:54:32,-0.1351
4,GPS_Auto_000004,806504.1683,2.343120e+06,-14.0468,-13.9252,0.0155,2026-10-02 09:54:33,-0.1216


In [19]:
def add_trajectory_and_speed(gps):
    gps = gps.copy()

        # Extract the label from point_id ("GPS_Auto_000001" -> "GPS_Auto", "GPS0001" -> "GPS")
    gps["trajectory"] = gps["point_id"].str.extract(
        r"^([A-Za-z]+(?:_[A-Za-z]+)*)",
        expand=False,
    )
    # Day 2 calls the boat track GPS_Auto; give it the same label as day 1
    gps["trajectory"] = gps["trajectory"].replace({"GPS_Auto": "TEST"})

    # Calculate horizontal speed within each trajectory
    ordered_gps = gps.sort_values(["trajectory", "datetime"])

    delta_easting = ordered_gps.groupby("trajectory")["easting"].diff()
    delta_northing = ordered_gps.groupby("trajectory")["northing"].diff()

    delta_time = (
        ordered_gps.groupby("trajectory")["datetime"]
        .diff()
        .dt.total_seconds()
    )

    distance = np.hypot(delta_easting, delta_northing)

    # Speed in metres per second
    speed_m_s = distance / delta_time.where(delta_time > 0)

    # Add the results to the original DataFrame order
    gps["speed_m_s"] = speed_m_s.reindex(gps.index)
    gps["speed_km_h"] = gps["speed_m_s"] * 3.6
    return gps


gps_D1 = add_trajectory_and_speed(gps_D1)
gps_D2 = add_trajectory_and_speed(gps_D2)

test_D1 = gps_D1[gps_D1["trajectory"] == "TEST"].sort_values("datetime")
test_D2 = gps_D2[gps_D2["trajectory"] == "TEST"].sort_values("datetime")

gps_points_D1 = gps_D1[gps_D1["trajectory"] == "GPS"].sort_values("datetime")
gps_points_D2 = gps_D2[gps_D2["trajectory"] == "GPS"].sort_values("datetime")

show = ["point_id", "trajectory", "datetime", "speed_m_s", "speed_km_h"]
display(gps_D1[show])
display(gps_D2[show])

# Both days in one table: the functions select the right day by date and time
gps = pd.concat([gps_D1, gps_D2], ignore_index=True)
gps_points = pd.concat([gps_points_D1, gps_points_D2], ignore_index=True)

,point_id,trajectory,datetime,speed_m_s,speed_km_h
0,GPS0001,GPS,2026-09-29 11:18:19,NaN,NaN
1,GPS0002,GPS,2026-09-29 11:18:40,0.074992,0.269972
2,GPS0003,GPS,2026-09-29 11:19:18,0.264426,0.951933
3,GPS0004,GPS,2026-09-29 11:19:56,0.362235,1.304045
4,GPS0005,GPS,2026-09-29 11:22:42,0.417235,1.502046
...,...,...,...,...,...
12898,TEST012676,TEST,2026-09-29 11:09:01,0.618595,2.226943
12899,TEST012677,TEST,2026-09-29 11:09:02,0.770201,2.772725
12900,TEST012678,TEST,2026-09-29 11:09:03,0.628138,2.261297
12901,TEST012679,TEST,2026-09-29 11:09:04,0.256624,0.923846


,point_id,trajectory,datetime,speed_m_s,speed_km_h
0,RTCM-Ref0017,RTCM,2026-10-02 09:52:57,NaN,NaN
1,GPS_Auto_000001,TEST,2026-10-02 09:54:30,NaN,NaN
2,GPS_Auto_000002,TEST,2026-10-02 09:54:31,0.328568,1.182845
3,GPS_Auto_000003,TEST,2026-10-02 09:54:32,0.368721,1.327395
4,GPS_Auto_000004,TEST,2026-10-02 09:54:33,0.475849,1.713056
...,...,...,...,...,...
6143,GPS0001,GPS,2026-10-02 11:43:30,NaN,NaN
6144,GPS0002,GPS,2026-10-02 11:44:11,0.021656,0.077960
6145,GPS0003,GPS,2026-10-02 11:44:36,0.018149,0.065336
6146,GPS0004,GPS,2026-10-02 11:45:17,0.076163,0.274187


In [25]:
cols = [
    "burst",        # Burst counter
    "ensemble",     # Ensemble counter (1-65536)
    "w",            # Velocity Beam1|X|East (m/s) orginally u renamed to w to reflect vertical velocities
    "u",            # Velocity Beam2|Y|North (m/s) originally v now renamed to u to reflect crossh shore velocities
    "v",            # Velocity Beam3|Z|Up (m/s) orginally w now renamed to v to reflect alongshore velocities
    "amp1",         # Amplitude Beam1 (counts)
    "amp2",         # Amplitude Beam2 (counts)
    "amp3",         # Amplitude Beam3 (counts)
    "snr1",         # SNR Beam1 (dB)
    "snr2",         # SNR Beam2 (dB)
    "snr3",         # SNR Beam3 (dB)
    "corr1",        # Correlation Beam1 (%)
    "corr2",        # Correlation Beam2 (%)
    "corr3",        # Correlation Beam3 (%)
    "pressure",     # Pressure (dbar)
    "analog1",      # Analog input 1
    "analog2",      # Analog input 2
    "checksum",     # Checksum (1 = failed)
]

df_SSC01 = pd.read_csv("SSC01.dat", sep=r"\s+", header=None, names=cols)
df_SSC02L = pd.read_csv("SSC02A.dat", sep=r"\s+", header=None, names=cols)
df_SSC02H = pd.read_csv("SSC02B.dat", sep=r"\s+", header=None, names=cols)

In [26]:
df_SSC01["t"] = df_SSC01.index / 16                     #Data from first field compain on 29 september
start = pd.Timestamp("2026-09-29 06:00:00")
df_SSC01["time"] = start + pd.to_timedelta(df_SSC01["t"], unit="s")

df_SSC02L["t"] = df_SSC02L.index / 16.                  #Data from second field compain on 2 oktober from the lowest sensor
start = pd.Timestamp("2026-10-02 06:00:00")
df_SSC02L["time"] = start + pd.to_timedelta(df_SSC02L["t"], unit="s")

df_SSC02H["t"] = df_SSC02H.index / 16                   #Data from second field compain on 2 oktober from the highest sensor
start = pd.Timestamp("2026-10-02 06:00:00")
df_SSC02H["time"] = start + pd.to_timedelta(df_SSC02H["t"], unit="s")

In [27]:
m = pd.read_csv("manoeuvresTimes.csv") #import of the manoeuvres times
manoeuvres = {k: v.split(";") for k, v in zip(m["id"], m["times"])}

# Both days in one table: the functions select the right day by date and time
gps = pd.concat([gps_D1, gps_D2], ignore_index=True)
gps_points = pd.concat([gps_points_D1, gps_points_D2], ignore_index=True)

# How each GPS point is drawn on the map: colour, marker shape, size (s) and legend name
GPS_STYLE = {
    "GPS0001": dict(color="black",  marker="^", label="Flag"),
    "GPS0002": dict(color="black",  marker="^", label="Flag"),
    "GPS0003": dict(color="purple", marker="*", s=220, label="Sensor"),
    "GPS0004": dict(color="black",  marker="^", label="Flag"),
}
GPS_DEFAULT = dict(color="0.6", marker="x", s=90, label="GPS point")   # any point not listed above

def select(ids):
    """Accepts one id ("B-11"), a list (["B-11", "B-12"]) or a whole type ("B")."""
    if isinstance(ids, str):
        ids = [ids]
    out = []
    for i in ids:
        if i in manoeuvres:
            out.append(i)
        else:
            matches = [k for k in manoeuvres if k.split("-")[0] == i]
            if not matches:
                raise KeyError(f"Unknown manoeuvre or type: {i}")
            out += matches
    return out

def sources(name, sensor="Both"):
    """Which dataframe(s) to use for a manoeuvre, from its series digit (the 3 in B-31)."""
    if name[-2] in "1234":
        return [(df_SSC01, "", "-")]
    out = []
    if sensor in ("Both", "Low"):
        out.append((df_SSC02L, "Low", "-."))
    if sensor in ("Both", "High"):
        out.append((df_SSC02H, "High", "--"))
    return out

def plot_track(ax, name, c, passes, before, after, norm, colour):
    """GPS track of the boat around one manoeuvre. c is the time of the first pass."""
    start = c - pd.Timedelta(seconds=before)
    end = c + pd.Timedelta(seconds=passes[-1] + after)

    test = gps[gps["datetime"].between(start, end) & (gps["trajectory"] == "TEST")].sort_values("datetime")
    if test.empty:
        print(f"No GPS track around {name} ({c:%Y-%m-%d %H:%M:%S}), not on the map")
        return [], []
    rel_t = (test["datetime"] - c).dt.total_seconds()
    ax.plot(test["easting"], test["northing"], color=colour, lw=1, label=name)
    ax.scatter(test["easting"], test["northing"], c=rel_t, cmap="viridis", norm=norm, s=15, zorder=3)

    # Position of the boat at the moment of each pass, interpolated along the track.
    inside = [p for p in passes if rel_t.min() <= p <= rel_t.max()]
    x, y = np.interp(inside, rel_t, test["easting"]), np.interp(inside, rel_t, test["northing"])
    ax.scatter(x, y, color="red", marker="x", s=100, linewidths=2, zorder=5)
    if inside:                              # name next to the first pass
        ax.annotate(name, (x[0], y[0]), xytext=(8, 8), textcoords="offset points", fontweight="bold")

    # GPS points measured during this manoeuvre get the colour of their time.
    if "gps_points" in globals():
        pts = gps_points[gps_points["datetime"].between(start, end)]
        ax.scatter(pts["easting"], pts["northing"], c=(pts["datetime"] - c).dt.total_seconds(),
                   cmap="viridis", norm=norm, marker="x", s=70, linewidths=1.5, zorder=4)
    return list(x), list(y)                 # where the boat was at each pass

import cv2, imageio
from IPython.display import Video

def plot_video(name, video, video_start, sensor="Both", before=120, after=120, col="analog1", fps=5, out=None):
    """Makes a video with the footage on the left and the sensor signal on the right.
    A red line moves through the plot in step with the footage.
    video_start is the clock time of the first frame of the video, e.g. "2026-09-29 10:15:02"."""
    t = manoeuvres[name]
    if isinstance(t, str):
        t = [t]
    offsets = pd.to_timedelta(t)
    passes = [(off - offsets[0]).total_seconds() for off in offsets]   # seconds after the first pass
    c = sources(name)[0][0]["time"].iloc[0].normalize() + offsets[0]   # clock time of the first pass
    end = passes[-1] + after

    # Open the video and jump to "before" seconds ahead of the passage
    cap = cv2.VideoCapture(video)
    vfps = cap.get(cv2.CAP_PROP_FPS)
    length = cap.get(cv2.CAP_PROP_FRAME_COUNT) / vfps
    first = (c - pd.Timestamp(video_start)).total_seconds() - before   # position in the video (s)
    if first < 0 or first + before + end > length:
        print(f"{name} ({c:%H:%M:%S}) is not fully inside this video "
              f"(needs {first:.0f} s to {first + before + end:.0f} s, video is {length:.0f} s long)")
        return
    cap.set(cv2.CAP_PROP_POS_MSEC, first * 1000)
    step = max(1, round(vfps / fps))        # use every step-th frame of the footage

    # The plot is drawn once; only the footage and the red line change per frame
    fig, (ax_v, ax) = plt.subplots(1, 2, figsize=(16, 4.8), dpi=100, layout="constrained",
                                   gridspec_kw={"width_ratios": [1, 1.2]})
    for data, suffix, ls in sources(name, sensor):
        win = data[(data["time"] >= c - pd.Timedelta(seconds=before)) &
                   (data["time"] <= c + pd.Timedelta(seconds=end))]
        ax.plot((win["time"] - c).dt.total_seconds(), win[col], ls=ls, lw=0.8, label=f"{name}{suffix}")
    for x in passes:
        ax.axvline(x, color="k", ls="--", lw=0.8)
    cursor = ax.axvline(-before, color="red", lw=1.5)
    ax.set_xlim(-before, end)
    ax.set_xlabel("Time relative to boat passage (s)")
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    ax_v.axis("off")
    im = None

    out = out or f"{name}_{col}.mp4"
    with imageio.get_writer(out, fps=vfps / step) as writer:
        i = 0
        while True:
            s = round(-before + i * step / vfps, 3)     # seconds relative to the passage
            ok, frame = cap.read()
            if s > end or not ok:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if frame.shape[0] > 720:        # smaller picture: faster and a smaller file
                k = 720 / frame.shape[0]
                frame = cv2.resize(frame, None, fx=k, fy=k)
            if im is None:
                im = ax_v.imshow(frame)
            else:
                im.set_data(frame)
            cursor.set_xdata([s, s])
            ax_v.set_title(f"{name}   {c + pd.Timedelta(seconds=s):%H:%M:%S}   ({s:+.1f} s)")
            fig.canvas.draw()
            writer.append_data(np.asarray(fig.canvas.buffer_rgba())[..., :3])
            for _ in range(step - 1):       # skip the frames in between
                cap.grab()
            i += 1
    cap.release()
    plt.close(fig)
    print(f"Saved {out}")
    return Video(out, embed=True)

#### Plotting manoeuvres with `plot_overlay`
`plot_overlay` plots the sensor signal around one or more boat passages, with time shown relative to the moment of passage. Next to it, the GPS track of the boat over the same period is shown on a map.

```python
plot_overlay("B-21", before=120, after=120, col="w")   # one manoeuvre
plot_overlay(["B-11", "B-12"], col="analog1")          # a few manoeuvres
plot_overlay("B", col="analog1")                       # every B manoeuvre
```
| Argument | What it does | Default |
|---|---|---|
| `before` | Seconds to plot before the manoeuvre | `120` |
| `after` | Seconds to plot after the manoeuvre | `120` |
| `col` | Column to plot, e.g. `"analog1"`, `"u"`, `"v"`, `"w"`, `"pressure"` | `"analog1"` |
| `sensor` | Which sensor to use for the second field day (see below) | `"Both"` |
| `show_map` | Show the GPS map next to the plot | `True` |


The `sensor` option only applies to data from the second field day (2 October).
- `sensor="Both"` → plots both the high and the low sensor
- `sensor="Low"` → plots only the low sensor
- `sensor="High"` → plots only the high sensor

In [37]:
plot_video("B2-54", "trim_above.mp4", "2026-10-02 10:34:48", sensor="Both", before=10, after=60, col="analog1", fps=12, out="B2-54_analog1.mp4")


Saved B2-54_analog1.mp4
